In [1]:
import xgboost as xgb
import pandas as pd
import src.train_utils as T
from sklearn.dummy import DummyRegressor

In [11]:
df = pd.read_csv('../datasets/exp_frp_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [12]:
train_df = T.add_time_lags(train_df, [1, 2, 3, 5, 7], ['delta_pm25_t'])
train_df = T.add_moving_averages(train_df, [3, 5, 7], ['delta_pm25_t'])

In [13]:
train_df = train_df.dropna()

In [14]:
train_df.columns

Index(['row', 'col', 'date', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t+1', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t-lag3',
       'delta_pm25_t-lag5', 'delta_pm25_t-lag7', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5', 'delta_pm25_t_ma7'],
      dtype='object')

In [15]:
base = []

feature_experiments = [
    ('persistence', base),
]

model=DummyRegressor(strategy='constant', constant=0)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: persistence
Fold 0
RMSE: 1.7836516750347196
Fold 1
RMSE: 6.286165987056864
Fold 2
RMSE: 5.223866135065733
Fold 3
RMSE: 9.354228728946818
Fold 4
RMSE: 14.504135042313575
Fold 5
RMSE: 2.214256839087764
Fold 6
RMSE: 1.003455618636538
Fold 7
RMSE: 1.0617824382856687
Fold 8
RMSE: 1.9852754348402382
Fold 9
RMSE: 4.290847572626642
    experiment  n_features  unweighted_rmse  weighted_rmse
0  persistence           0         4.770767       6.657028


In [16]:
base = ['pm25_t', 'u_wind_10m_t', 'v_wind_10m_t', 'delta_pm25_t', 'precip_sum_t', 'dew_temp_2m_t', 'temp_2m_t', 'frp_t',
        'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7']


params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + ma3', base + ['delta_pm25_t_ma3']),
    ('base + ma3 + ma5', base + ['delta_pm25_t_ma3', 'delta_pm25_t_ma5']),
    ('base + ma3 + ma5 + ma7', base + ['delta_pm25_t_ma3', 'delta_pm25_t_ma5', 'delta_pm25_t_ma7']),
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7529297381960114
Fold 1
RMSE: 5.441201793880853
Fold 2
RMSE: 5.146084236047798
Fold 3
RMSE: 8.767369416661987
Fold 4
RMSE: 13.08596401664314
Fold 5
RMSE: 2.1945884598774437
Fold 6
RMSE: 1.0018822086810864
Fold 7
RMSE: 1.0807013207631802
Fold 8
RMSE: 1.9406109284643045
Fold 9
RMSE: 4.175479950947372
Running experiment: base + ma3
Fold 0
RMSE: 1.7077169213508594
Fold 1
RMSE: 5.406432576237439
Fold 2
RMSE: 5.118704672612572
Fold 3
RMSE: 8.674324482004241
Fold 4
RMSE: 13.25182150024823
Fold 5
RMSE: 2.157762667066412
Fold 6
RMSE: 0.9799702694617999
Fold 7
RMSE: 1.052851962819424
Fold 8
RMSE: 1.8898311306875402
Fold 9
RMSE: 4.080155960384204
Running experiment: base + ma3 + ma5
Fold 0
RMSE: 1.700131928639953
Fold 1
RMSE: 5.331446797901701
Fold 2
RMSE: 5.124908377586923
Fold 3
RMSE: 8.64552756769358
Fold 4
RMSE: 13.2229383242663
Fold 5
RMSE: 2.148676142631054
Fold 6
RMSE: 0.9582656616288703
Fold 7
RMSE: 1.044079307901115
Fold 8
RMSE: 1.898098892450767
F

In [17]:
base = ['pm25_t', 'u_wind_10m_t', 'v_wind_10m_t', 'delta_pm25_t', 'precip_sum_t', 'dew_temp_2m_t', 'temp_2m_t', 'frp_t',
        'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + lag1', base + ['delta_pm25_t-lag1']),
    ('base + lag1 + lag2', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2']),
    ('base + lag1 + lag2 + lag3', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t-lag3']),
    ('base + lag1 + lag2 + lag3 + lag5', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t-lag3', 'delta_pm25_t-lag5']),
    ('base + lag1 + lag2 + lag3 + lag5 + lag7', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t-lag3', 'delta_pm25_t-lag5', 'delta_pm25_t-lag7']),
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7529297381960114
Fold 1
RMSE: 5.441201793880853
Fold 2
RMSE: 5.146084236047798
Fold 3
RMSE: 8.767369416661987
Fold 4
RMSE: 13.08596401664314
Fold 5
RMSE: 2.1945884598774437
Fold 6
RMSE: 1.0018822086810864
Fold 7
RMSE: 1.0807013207631802
Fold 8
RMSE: 1.9406109284643045
Fold 9
RMSE: 4.175479950947372
Running experiment: base + lag1
Fold 0
RMSE: 1.7384616554699146
Fold 1
RMSE: 5.49234986920307
Fold 2
RMSE: 5.147736150524211
Fold 3
RMSE: 8.729238057008104
Fold 4
RMSE: 13.126711634386314
Fold 5
RMSE: 2.1936179886658085
Fold 6
RMSE: 1.0015340599492735
Fold 7
RMSE: 1.0750803163046985
Fold 8
RMSE: 1.937230035826446
Fold 9
RMSE: 4.112190388622544
Running experiment: base + lag1 + lag2
Fold 0
RMSE: 1.7339029097491743
Fold 1
RMSE: 5.452367953998336
Fold 2
RMSE: 5.134779357550911
Fold 3
RMSE: 8.686176696297936
Fold 4
RMSE: 13.129786394078096
Fold 5
RMSE: 2.195440842394402
Fold 6
RMSE: 1.0001474291922643
Fold 7
RMSE: 1.0714636419165877
Fold 8
RMSE: 1.94892700

In [18]:
base = ['pm25_t', 'u_wind_10m_t', 'v_wind_10m_t', 'delta_pm25_t', 'precip_sum_t', 'dew_temp_2m_t', 'temp_2m_t', 'frp_t',
        'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + lag1 + lag2', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2']),
    ('base + ma3 + ma5', base + ['delta_pm25_t_ma3', 'delta_pm25_t_ma5']),
    ('base + lag1 + lag2 + ma3 + ma5', base + ['delta_pm25_t-lag1', 'delta_pm25_t-lag2'] + ['delta_pm25_t_ma3', 'delta_pm25_t_ma5'])
    
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7529297381960114
Fold 1
RMSE: 5.441201793880853
Fold 2
RMSE: 5.146084236047798
Fold 3
RMSE: 8.767369416661987
Fold 4
RMSE: 13.08596401664314
Fold 5
RMSE: 2.1945884598774437
Fold 6
RMSE: 1.0018822086810864
Fold 7
RMSE: 1.0807013207631802
Fold 8
RMSE: 1.9406109284643045
Fold 9
RMSE: 4.175479950947372
Running experiment: base + lag1 + lag2
Fold 0
RMSE: 1.7339029097491743
Fold 1
RMSE: 5.452367953998336
Fold 2
RMSE: 5.134779357550911
Fold 3
RMSE: 8.686176696297936
Fold 4
RMSE: 13.129786394078096
Fold 5
RMSE: 2.195440842394402
Fold 6
RMSE: 1.0001474291922643
Fold 7
RMSE: 1.0714636419165877
Fold 8
RMSE: 1.9489270040658846
Fold 9
RMSE: 4.113611819914457
Running experiment: base + ma3 + ma5
Fold 0
RMSE: 1.700131928639953
Fold 1
RMSE: 5.331446797901701
Fold 2
RMSE: 5.124908377586923
Fold 3
RMSE: 8.64552756769358
Fold 4
RMSE: 13.2229383242663
Fold 5
RMSE: 2.148676142631054
Fold 6
RMSE: 0.9582656616288703
Fold 7
RMSE: 1.044079307901115
Fold 8
RMSE: 1.8980988

In [20]:
train_df.columns

Index(['row', 'col', 'date', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t', 'delta_pm25_t+1', 'delta_pm25_t',
       'frp_t_mean_3x3', 'frp_t_mean_5x5', 'frp_t_mean_7x7',
       'delta_pm25_t-lag1', 'delta_pm25_t-lag2', 'delta_pm25_t-lag3',
       'delta_pm25_t-lag5', 'delta_pm25_t-lag7', 'delta_pm25_t_ma3',
       'delta_pm25_t_ma5', 'delta_pm25_t_ma7'],
      dtype='object')

In [21]:
exp_df = train_df.drop(['delta_pm25_t-lag3', 'delta_pm25_t-lag5', 'delta_pm25_t-lag7', 'delta_pm25_t_ma7'], axis=1)

In [22]:
exp_df.to_csv('exp_time_lags_dataset.csv', index=False)